In [ ]:
#ver 2

In [53]:
"""
Bus Fleet Dispatch and Frequency Optimisation
Small-sized Gurobi benchmark using frequency-based allocation.

Updated formulation:
- Decision is frequency / number of dispatches per service-period.
- Headway is derived from frequency: h = period_length / frequency.
- Demand is allocated across dispatches: demand_per_dispatch = demand / frequency.
- Overcrowding is calculated based on whether total demand exceeds total dispatch capacity.
- Fleet requirement is estimated from cycle time and selected frequency.
"""

import math
import time
from gurobipy import Model, GRB, quicksum


# ============================================================
# 1. Create small-sized Tampines example instance
# ============================================================

def create_small_instance():
    """
    Small illustrative instance.

    Replace the sample values with your processed LTA data:
    - D[i,t]: passenger demand from tap-in / passenger volume data
    - headway_options: observed or selected headway values from bus arrival data
    - cycle_time[i]: estimated round-trip cycle time for each service
    - q[i]: bus capacity, e.g. 70 for single-deck, 100 for double-deck
    """

    services = ["8", "18", "28", "38", "69"]

    periods = ["Morning_Peak", "Off_Peak", "Evening_Peak"]

    # Each time period is 2 hours = 120 minutes
    # Or maybe we can change it to 1 hour and then multiply by 2? since the LTA data is given per hour
    L = {
        "Morning_Peak": 120,
        "Off_Peak": 120,
        "Evening_Peak": 120,
    }

    # Available fleet during each time period
    # This is the number of physical buses available for the selected services. I got these numbers as a rough estimation
    available_fleet = {
        "Morning_Peak": 500,
        "Off_Peak": 300,
        "Evening_Peak": 500,
    }

    # Estimated passenger demand by service and period
    # Replace with LTA passenger volume data
    D = {
        ("8", "Morning_Peak"): 850,
        ("8", "Off_Peak"): 420,
        ("8", "Evening_Peak"): 780,

        ("18", "Morning_Peak"): 520,
        ("18", "Off_Peak"): 260,
        ("18", "Evening_Peak"): 500,

        ("28", "Morning_Peak"): 920,
        ("28", "Off_Peak"): 460,
        ("28", "Evening_Peak"): 900,

        ("38", "Morning_Peak"): 430,
        ("38", "Off_Peak"): 200,
        ("38", "Evening_Peak"): 410,

        ("69", "Morning_Peak"): 760,
        ("69", "Off_Peak"): 370,
        ("69", "Evening_Peak"): 730,
    }

    # Bus Capacity
    # 70 for single-deck and 100 for double-deck
    q = {
        "8": 100,
        "18": 70,
        "28": 100,
        "38": 70,
        "69": 100,
    }

    # Estimated round-trip cycle time in minutes
    # Used only to estimate physical buses required for a selected frequency
    cycle_time = {
        "8": 65,
        "18": 55,
        "28": 70,
        "38": 50,
        "69": 60,
    }

    # Candidate headway options
    # Comes from LTA bus arrival data (arrival time of bus 2 - arrival time of bus 1)
    # Example: If observed headways during AM peak are around 4, 5, 6, and 8 minutes, use [4, 5, 6, 8]
    headway_options = {}

    for i in services:
        for t in periods:
            if t in ["Morning_Peak", "Evening_Peak"]:
                headway_options[i, t] = [3, 4, 5, 7]
            else:
                headway_options[i, t] = [5, 6, 7.5, 8]

    return services, periods, L, available_fleet, D, q, cycle_time, headway_options


# ============================================================
# 2. Solve exact small-sized model using Gurobi
# ============================================================

def solve_frequency_based_gurobi(
    services,
    periods,
    L,
    available_fleet,
    D,
    q,
    cycle_time,
    headway_options,
    alpha=0.40,
    beta=0.50,
    gamma=0.10,
    time_limit=60,
    verbose=True,
):
    """
    Gurobi model.

    Binary decision variable:
        y[i,t,h] = 1 if service i in period t uses headway option h.

    Derived values:
        frequency = L[t] / h
        demand_per_dispatch = D[i,t] / frequency
        total_capacity = frequency * q[i]
        overcrowding = max(0, D[i,t] - total_capacity)
        waiting/headway penalty = (h - 5)^2 + 2
        buses_required = ceil(cycle_time[i] / h)

    Objective:
        Minimise weighted sum of:
        1. passenger-weighted waiting/headway penalty
        2. overcrowding penalty
        3. number of physical buses required
    """

    model = Model("frequency_based_bus_dispatch")

    if not verbose:
        model.Params.OutputFlag = 0

    model.Params.TimeLimit = time_limit

    # ------------------------------------------------------------
    # Precompute values for every service-period-headway option
    # ------------------------------------------------------------

    frequency = {}
    buses_required = {}
    demand_per_dispatch = {}
    total_capacity = {}
    overcrowding = {}
    overcrowding_penalty = {}
    headway_penalty = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:

                # Frequency = number of departures in the 2-hour period, which is also the no. of buses
                frequency[i, t, h] = L[t] / h

                # Demand allocated to each dispatch, since we only have the total tap-in value for each hour
                demand_per_dispatch[i, t, h] = D[i, t] / frequency[i, t, h]

                # Total capacity across all dispatches. this is essentially no. of buses * capacity of 1 bus
                total_capacity[i, t, h] = frequency[i, t, h] * q[i]

                # Overcrowding amount = when total demand exceeds total capacity
                overcrowding[i, t, h] = max(
                    0,
                    D[i, t] - total_capacity[i, t, h]
                )

                # Squared overcrowding penalty
                overcrowding_penalty[i, t, h] = overcrowding[i, t, h] ** 2

                # Quadratic headway penalty
                headway_penalty[i, t, h] = (h - 5) ** 2 + 2

                # Physical buses required to sustain that headway
                # It estimates fleet requirement from cycle time / selected headway
                buses_required[i, t, h] = math.ceil(cycle_time[i] / h)

    # ------------------------------------------------------------
    # Decision variables
    # ------------------------------------------------------------

    y = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:
                y[i, t, h] = model.addVar(
                    vtype=GRB.BINARY,
                    name=f"choose_{i}_{t}_h{h}"
                )

    model.update()

    # ------------------------------------------------------------
    # Constraint 1:
    # Choose exactly one frequency/headway option for each service-period
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            model.addConstr(
                quicksum(y[i, t, h] for h in headway_options[i, t]) == 1,
                name=f"choose_one_option_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 2:
    # Fleet availability during each time period
    # ------------------------------------------------------------

    for t in periods:
        model.addConstr(
            quicksum(
                buses_required[i, t, h] * y[i, t, h]
                for i in services
                for h in headway_options[i, t]
            ) <= available_fleet[t],
            name=f"fleet_limit_{t}"
        )

    # ------------------------------------------------------------
    # Normalisation
    # This avoids one objective component dominating only because of scale.
    # ------------------------------------------------------------

    max_waiting_component = sum(
        D[i, t] * max(headway_penalty[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_overcrowding_component = sum(
        max(overcrowding_penalty[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_bus_component = sum(
        max(buses_required[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_waiting_component = max(max_waiting_component, 1)
    max_overcrowding_component = max(max_overcrowding_component, 1)
    max_bus_component = max(max_bus_component, 1)

    # ------------------------------------------------------------
    # Objective components
    # ------------------------------------------------------------

    waiting_component = quicksum(
        D[i, t] * headway_penalty[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    overcrowding_component = quicksum(
        overcrowding_penalty[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    bus_component = quicksum(
        buses_required[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    model.setObjective(
        alpha * waiting_component / max_waiting_component
        + beta * overcrowding_component / max_overcrowding_component
        + gamma * bus_component / max_bus_component,
        GRB.MINIMIZE
    )

    # ------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------

    start_time = time.time()
    model.optimize()
    runtime = time.time() - start_time

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print("No feasible solution found.")
        print("Gurobi status:", model.Status)
        return None

    # ------------------------------------------------------------
    # Extract solution
    # ------------------------------------------------------------

    solution = []

    raw_waiting = 0
    raw_overcrowding = 0
    raw_buses = 0

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:
                if y[i, t, h].X > 0.5:

                    row = {
                        "service": i,
                        "period": t,
                        "headway_min": h,
                        "frequency_departures": frequency[i, t, h],
                        "buses_required": buses_required[i, t, h],
                        "demand": D[i, t],
                        "demand_per_dispatch": demand_per_dispatch[i, t, h],
                        "capacity_per_dispatch": q[i],
                        "total_capacity": total_capacity[i, t, h],
                        "overcrowding": overcrowding[i, t, h],
                        "headway_penalty": headway_penalty[i, t, h],
                        "overcrowding_penalty": overcrowding_penalty[i, t, h],
                    }

                    solution.append(row)

                    raw_waiting += D[i, t] * headway_penalty[i, t, h]
                    raw_overcrowding += overcrowding_penalty[i, t, h]
                    raw_buses += buses_required[i, t, h]

    result = {
        "status": model.Status,
        "objective": model.ObjVal,
        "runtime_seconds": runtime,
        "mip_gap": model.MIPGap,
        "raw_waiting_component": raw_waiting,
        "raw_overcrowding_component": raw_overcrowding,
        "raw_bus_component": raw_buses,
        "solution": solution,
    }

    return result


# ============================================================
# 3. Print solution
# ============================================================

def print_solution(result):
    if result is None:
        return

    print("\n==============================")
    print("GUROBI SMALL-SIZED BENCHMARK")
    print("==============================")
    print(f"Objective value:          {result['objective']:.6f}")
    print(f"Runtime:                  {result['runtime_seconds']:.4f} seconds")
    print(f"MIP gap:                  {result['mip_gap']:.6f}")
    print(f"Raw waiting component:    {result['raw_waiting_component']:.2f}")
    print(f"Raw overcrowding penalty: {result['raw_overcrowding_component']:.2f}")
    print(f"Raw buses required:       {result['raw_bus_component']:.0f}")

    print("\nSelected frequency-based dispatch plan:")
    print("=" * 150)

    header = (
        f"{'Service':<8} "
        f"{'Period':<15} "
        f"{'Headway':<10} "
        f"{'Freq':<10} "
        f"{'Buses':<8} "
        f"{'Demand':<10} "
        f"{'Demand/Bus':<13} "
        f"{'Cap/Bus':<10} "
        f"{'Total Cap':<12} "
        f"{'Overcrowd':<12} "
        f"{'H Penalty':<10}"
    )

    print(header)
    print("-" * 150)

    for row in result["solution"]:
        print(
            f"{row['service']:<8} "
            f"{row['period']:<15} "
            f"{row['headway_min']:<10.2f} "
            f"{row['frequency_departures']:<10.2f} "
            f"{row['buses_required']:<8} "
            f"{row['demand']:<10.0f} "
            f"{row['demand_per_dispatch']:<13.2f} "
            f"{row['capacity_per_dispatch']:<10.0f} "
            f"{row['total_capacity']:<12.2f} "
            f"{row['overcrowding']:<12.2f} "
            f"{row['headway_penalty']:<10.2f}"
        )

    print("=" * 150)


# ============================================================
# 4. Simple greedy benchmark
# ============================================================

def greedy_frequency_benchmark(
    services,
    periods,
    L,
    available_fleet,
    D,
    q,
    cycle_time,
    headway_options,
    alpha=0.40,
    beta=0.50,
    gamma=0.10,
):
    """
    Simple benchmark heuristic:
    For each service-period, choose the locally best option,
    then repair if fleet availability is exceeded.

    This is not guaranteed optimal.
    It is useful as a baseline to compare against Gurobi.
    """

    def option_cost(i, t, h):
        freq = L[t] / h
        total_cap = freq * q[i]
        over = max(0, D[i, t] - total_cap)
        over_pen = over ** 2
        h_pen = (h - 5) ** 2 + 2
        buses = math.ceil(cycle_time[i] / h)

        return alpha * D[i, t] * h_pen + beta * over_pen + gamma * buses

    selected = {}

    # Step 1: choose locally cheapest option
    for i in services:
        for t in periods:
            best_h = min(
                headway_options[i, t],
                key=lambda h: option_cost(i, t, h)
            )
            selected[i, t] = best_h

    # Step 2: repair fleet violations by increasing headway
    for t in periods:
        while True:
            used_buses = sum(
                math.ceil(cycle_time[i] / selected[i, t])
                for i in services
            )

            if used_buses <= available_fleet[t]:
                break

            best_service_to_relax = None
            lowest_cost_increase = float("inf")

            for i in services:
                current_h = selected[i, t]
                sorted_options = sorted(headway_options[i, t])

                # Try moving to the next larger headway
                larger_options = [h for h in sorted_options if h > current_h]

                if not larger_options:
                    continue

                next_h = larger_options[0]

                old_cost = option_cost(i, t, current_h)
                new_cost = option_cost(i, t, next_h)
                cost_increase = new_cost - old_cost

                if cost_increase < lowest_cost_increase:
                    lowest_cost_increase = cost_increase
                    best_service_to_relax = i

            if best_service_to_relax is None:
                raise RuntimeError(
                    f"Cannot repair fleet violation for period {t}. "
                    f"Increase available fleet or allow larger headways."
                )

            current_h = selected[best_service_to_relax, t]
            larger_options = [
                h for h in sorted(headway_options[best_service_to_relax, t])
                if h > current_h
            ]

            selected[best_service_to_relax, t] = larger_options[0]

    total_cost = 0
    total_buses = 0

    for i in services:
        for t in periods:
            h = selected[i, t]
            total_cost += option_cost(i, t, h)
            total_buses += math.ceil(cycle_time[i] / h)

    return selected, total_cost, total_buses


# ============================================================
# 5. Complexity demonstration
# ============================================================

def complexity_demo(services, periods, headway_options):
    """
    Shows how the number of combinations increases.

    If each service-period has several frequency/headway options,
    total combinations = product of number of options across all service-periods.
    """

    total_combinations = 1

    for i in services:
        for t in periods:
            total_combinations *= len(headway_options[i, t])

    print("\n==============================")
    print("COMPUTATIONAL COMPLEXITY")
    print("==============================")
    print(f"Number of services:       {len(services)}")
    print(f"Number of time periods:   {len(periods)}")
    print(f"Total service-periods:    {len(services) * len(periods)}")
    print(f"Total combinations:       {total_combinations:,}")

    print(
        "\nThis grows exponentially because the model must choose one "
        "frequency/headway option for every service-period pair."
    )


# ============================================================
# 6. Main execution
# ============================================================

if __name__ == "__main__":

    instance = create_small_instance()

    services, periods, L, available_fleet, D, q, cycle_time, headway_options = instance

    result = solve_frequency_based_gurobi(
        services,
        periods,
        L,
        available_fleet,
        D,
        q,
        cycle_time,
        headway_options,
        alpha=0.40,
        beta=0.50,
        gamma=0.10,
        time_limit=60,
        verbose=True,
    )

    print_solution(result)

    selected, greedy_cost, greedy_buses = greedy_frequency_benchmark(
        services,
        periods,
        L,
        available_fleet,
        D,
        q,
        cycle_time,
        headway_options,
        alpha=0.40,
        beta=0.50,
        gamma=0.10,
    )

    print("\n==============================")
    print("GREEDY BENCHMARK")
    print("==============================")
    print(f"Greedy cost:        {greedy_cost:.2f}")
    print(f"Greedy buses used:  {greedy_buses}")
    print("\nGreedy selected headways:")

    for key, value in selected.items():
        print(f"{key}: {value} minutes")

    complexity_demo(services, periods, headway_options)

Set parameter TimeLimit to value 60
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60

Optimize a model with 18 rows, 60 columns and 120 nonzeros (Min)
Model fingerprint: 0xf8acf152
Model has 60 linear objective coefficients
Variable types: 0 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [6e-03, 5e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]

Found heuristic solution: objective 0.2952614
Presolve removed 18 rows and 60 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 0.182391 0.295261 

Optimal solution fou

In [67]:
#ver 3
#================================

"""
Bus Fleet Dispatch and Frequency Optimisation
Small-sized Gurobi benchmark with frequency-based demand allocation.

This version explicitly includes:

D_{i,s,t} = D_{s,t} * f_{i,t} / sum_{j in S_s} f_{j,t}

where:
    D_{s,t}     = total passenger demand at stop s during period t
    D_{i,s,t}   = demand allocated to service i at stop s during period t
    f_{i,t}     = frequency of service i during period t
    S_s         = set of services serving stop s

Gurobi solves this as a small nonconvex MIQCP benchmark.

Install:
    pip install gurobipy
"""

import math
import time
from gurobipy import Model, GRB, quicksum


# ============================================================
# 1. Small example data
# ============================================================

def create_small_instance():
    """
    Small illustrative Tampines-style example.

    Replace these values with your processed LTA data later.
    """

    # Selected bus services
    services = ["291", "292", "293", "296"]

    # Selected bus stops
    stops = ["75009", "76059", "76101"]

    # Time periods
    periods = ["Morning_Peak", "Off_Peak", "Evening_Peak"]

    # Length of each period in minutes
    L = {
        "Morning_Peak": 120,
        "Off_Peak": 120,
        "Evening_Peak": 120,
    }

    # Available physical buses in each period
    available_fleet = {
        "Morning_Peak": 40,
        "Off_Peak": 30,
        "Evening_Peak": 50,
    }

    # Services serving each stop
    # S_s in your formulation
    services_at_stop = {
        "75009": ["291", "292", "293", "296"],
        "76059": ["291", "292"],
        "76101": ["293", "296"],
    }

    # Stop-level passenger demand D_{s,t}
    # Example: total tap-in demand at each stop during each period
    stop_demand = {
        ("75009", "Morning_Peak"): 2800,
        ("75009", "Off_Peak"): 1200,
        ("75009", "Evening_Peak"): 2500,

        ("76059", "Morning_Peak"): 900,
        ("76059", "Off_Peak"): 400,
        ("76059", "Evening_Peak"): 850,

        ("76101", "Morning_Peak"): 700,
        ("76101", "Off_Peak"): 350,
        ("76101", "Evening_Peak"): 650,
    }

    # Bus capacity
    # Use 70 for single-deck, 100 for double-deck
    bus_capacity = {
        "291": 70,
        "292": 70,
        "293": 100,
        "296": 70,
    }

    # Round-trip cycle time in minutes
    # Used to estimate physical buses required
    cycle_time = {
        "291": 55,
        "292": 50,
        "293": 60,
        "296": 45,
    }

    # Candidate headways in minutes
    # Frequency will be f = L / h
    headway_options = {}

    for i in services:
        for t in periods:
            if t in ["Morning_Peak", "Evening_Peak"]:
                headway_options[i, t] = [3, 4, 5, 7]
            else:
                headway_options[i, t] = [5, 6, 7.5, 8]

    return (
        services,
        stops,
        periods,
        L,
        available_fleet,
        services_at_stop,
        stop_demand,
        bus_capacity,
        cycle_time,
        headway_options,
    )


# ============================================================
# 2. Gurobi model
# ============================================================

def solve_gurobi_frequency_allocation(
    services,
    stops,
    periods,
    L,
    available_fleet,
    services_at_stop,
    stop_demand,
    bus_capacity,
    cycle_time,
    headway_options,
    alpha=0.40,
    beta=0.50,
    gamma=0.10,
    time_limit=120,
    verbose=True,
):
    """
    Solves the small-sized benchmark using Gurobi.

    Main decision:
        Choose one headway option for each service-period.

    Frequency:
        f_{i,t} = L_t / h_{i,t}

    Frequency-based allocation:
        D_{i,s,t} = D_{s,t} * f_{i,t} / sum_{j in S_s} f_{j,t}

    Implemented as:
        D_{i,s,t} * total_frequency_{s,t}
        =
        D_{s,t} * f_{i,t}

    This is a bilinear equality, so NonConvex=2 is required.
    """

    model = Model("frequency_based_demand_allocation")

    if not verbose:
        model.Params.OutputFlag = 0

    model.Params.TimeLimit = time_limit
    model.Params.NonConvex = 2

    # ------------------------------------------------------------
    # Precompute option-level values
    # ------------------------------------------------------------

    frequency_option = {}
    headway_penalty_option = {}
    buses_required_option = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:

                # Frequency = number of departures in the period
                frequency_option[i, t, h] = L[t] / h

                # Headway penalty
                headway_penalty_option[i, t, h] = (h - 5) ** 2 + 2

                # Physical buses required to sustain the selected headway
                buses_required_option[i, t, h] = math.ceil(cycle_time[i] / h)

    # ------------------------------------------------------------
    # Decision variables
    # ------------------------------------------------------------

    # y[i,t,h] = 1 if service i in period t chooses headway h
    y = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:
                y[i, t, h] = model.addVar(
                    vtype=GRB.BINARY,
                    name=f"y_{i}_{t}_h{h}"
                )

    # Chosen frequency f[i,t]
    f = {}

    # Chosen buses required b[i,t]
    b = {}

    # Total frequency at stop s in period t
    total_freq = {}

    # Allocated demand D_alloc[i,s,t]
    D_alloc = {}

    # Total service demand after summing across stops
    service_demand = {}

    # Total service capacity
    service_capacity = {}

    # Overcrowding amount
    overcrowding = {}

    model.update()

    for i in services:
        for t in periods:
            f[i, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"frequency_{i}_{t}"
            )

            b[i, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"buses_required_{i}_{t}"
            )

            service_demand[i, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"service_demand_{i}_{t}"
            )

            service_capacity[i, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"service_capacity_{i}_{t}"
            )

            overcrowding[i, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"overcrowding_{i}_{t}"
            )

    for s in stops:
        for t in periods:
            total_freq[s, t] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"total_frequency_{s}_{t}"
            )

            for i in services_at_stop[s]:
                D_alloc[i, s, t] = model.addVar(
                    lb=0,
                    vtype=GRB.CONTINUOUS,
                    name=f"D_alloc_{i}_{s}_{t}"
                )

    model.update()

    # ------------------------------------------------------------
    # Constraint 1:
    # Choose exactly one headway option per service-period
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            model.addConstr(
                quicksum(y[i, t, h] for h in headway_options[i, t]) == 1,
                name=f"choose_one_headway_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 2:
    # Link chosen headway to frequency and buses required
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            model.addConstr(
                f[i, t]
                ==
                quicksum(
                    frequency_option[i, t, h] * y[i, t, h]
                    for h in headway_options[i, t]
                ),
                name=f"link_frequency_{i}_{t}"
            )

            model.addConstr(
                b[i, t]
                ==
                quicksum(
                    buses_required_option[i, t, h] * y[i, t, h]
                    for h in headway_options[i, t]
                ),
                name=f"link_buses_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 3:
    # Total frequency at each stop
    #
    # total_freq_{s,t} = sum_{j in S_s} f_{j,t}
    # ------------------------------------------------------------

    for s in stops:
        for t in periods:
            model.addConstr(
                total_freq[s, t]
                ==
                quicksum(f[j, t] for j in services_at_stop[s]),
                name=f"total_frequency_{s}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 4:
    # Frequency-based demand allocation
    #
    # D_{i,s,t} = D_{s,t} * f_{i,t} / sum_{j in S_s} f_{j,t}
    #
    # Implemented as:
    # D_alloc[i,s,t] * total_freq[s,t] = stop_demand[s,t] * f[i,t]
    #
    # This is the key part that was missing before.
    # ------------------------------------------------------------

    for s in stops:
        for t in periods:
            for i in services_at_stop[s]:
                model.addQConstr(
                    D_alloc[i, s, t] * total_freq[s, t]
                    ==
                    stop_demand[s, t] * f[i, t],
                    name=f"freq_based_allocation_{i}_{s}_{t}"
                )

    # ------------------------------------------------------------
    # Optional consistency check:
    # Sum of allocated demand at each stop equals total stop demand
    # ------------------------------------------------------------

    for s in stops:
        for t in periods:
            model.addConstr(
                quicksum(D_alloc[i, s, t] for i in services_at_stop[s])
                ==
                stop_demand[s, t],
                name=f"allocated_demand_balance_{s}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 5:
    # Service-level total demand
    #
    # D_{i,t} = sum_s D_{i,s,t}
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            served_stops = [
                s for s in stops
                if i in services_at_stop[s]
            ]

            model.addConstr(
                service_demand[i, t]
                ==
                quicksum(D_alloc[i, s, t] for s in served_stops),
                name=f"service_demand_sum_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 6:
    # Capacity and overcrowding
    #
    # Capacity_{i,t} = f_{i,t} * q_i
    # O_{i,t} >= D_{i,t} - Capacity_{i,t}
    # O_{i,t} >= 0
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            model.addConstr(
                service_capacity[i, t]
                ==
                bus_capacity[i] * f[i, t],
                name=f"service_capacity_{i}_{t}"
            )

            model.addConstr(
                overcrowding[i, t]
                >=
                service_demand[i, t] - service_capacity[i, t],
                name=f"overcrowding_def_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 7:
    # Fleet availability
    # ------------------------------------------------------------

    for t in periods:
        model.addConstr(
            quicksum(b[i, t] for i in services) <= available_fleet[t],
            name=f"fleet_limit_{t}"
        )

    # ------------------------------------------------------------
    # Objective components
    # ------------------------------------------------------------

    waiting_component = quicksum(
        stop_demand[s, t]
        *
        quicksum(
            headway_penalty_option[i, t, h] * y[i, t, h]
            for i in services_at_stop[s]
            for h in headway_options[i, t]
        )
        /
        len(services_at_stop[s])
        for s in stops
        for t in periods
    )

    # Squared overcrowding penalty
    overcrowding_component = quicksum(
        overcrowding[i, t] * overcrowding[i, t]
        for i in services
        for t in periods
    )

    bus_component = quicksum(
        b[i, t]
        for i in services
        for t in periods
    )

    # Normalisation values
    max_waiting = 1 + sum(
        stop_demand[s, t] * 20
        for s in stops
        for t in periods
    )

    max_overcrowding = 1 + sum(
        stop_demand[s, t] ** 2
        for s in stops
        for t in periods
    )

    max_buses = 1 + sum(
        available_fleet[t]
        for t in periods
    )

    model.setObjective(
        alpha * waiting_component / max_waiting
        +
        beta * overcrowding_component / max_overcrowding
        +
        gamma * bus_component / max_buses,
        GRB.MINIMIZE
    )

    # ------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------

    start_time = time.time()
    model.optimize()
    runtime = time.time() - start_time

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print("No feasible solution found.")
        print("Gurobi status:", model.Status)
        return None

    # ------------------------------------------------------------
    # Extract results
    # ------------------------------------------------------------

    result = {
        "status": model.Status,
        "objective": model.ObjVal,
        "runtime": runtime,
        "mip_gap": model.MIPGap,
        "selected_headways": [],
        "allocated_demand": [],
        "service_summary": [],
    }

    for i in services:
        for t in periods:
            selected_h = None

            for h in headway_options[i, t]:
                if y[i, t, h].X > 0.5:
                    selected_h = h
                    break

            result["selected_headways"].append({
                "service": i,
                "period": t,
                "headway": selected_h,
                "frequency": f[i, t].X,
                "buses_required": b[i, t].X,
            })

    for s in stops:
        for t in periods:
            for i in services_at_stop[s]:
                result["allocated_demand"].append({
                    "stop": s,
                    "period": t,
                    "service": i,
                    "stop_demand": stop_demand[s, t],
                    "service_frequency": f[i, t].X,
                    "total_stop_frequency": total_freq[s, t].X,
                    "allocated_demand": D_alloc[i, s, t].X,
                })

    for i in services:
        for t in periods:
            result["service_summary"].append({
                "service": i,
                "period": t,
                "frequency": f[i, t].X,
                "service_demand": service_demand[i, t].X,
                "service_capacity": service_capacity[i, t].X,
                "overcrowding": overcrowding[i, t].X,
                "buses_required": b[i, t].X,
            })

    return result


# ============================================================
# 3. Print results
# ============================================================

def print_results(result):
    if result is None:
        return

    print("\n==============================")
    print("GUROBI SOLUTION SUMMARY")
    print("==============================")
    print(f"Objective value: {result['objective']:.6f}")
    print(f"Runtime:         {result['runtime']:.4f} seconds")
    print(f"MIP gap:         {result['mip_gap']:.6f}")

    print("\nSelected headways and frequencies:")
    print("-" * 80)
    print(
        f"{'Service':<10} {'Period':<15} {'Headway':<10} "
        f"{'Frequency':<12} {'Buses Req.':<12}"
    )

    for row in result["selected_headways"]:
        print(
            f"{row['service']:<10} {row['period']:<15} "
            f"{row['headway']:<10.2f} {row['frequency']:<12.2f} "
            f"{row['buses_required']:<12.0f}"
        )

    print("\nFrequency-based demand allocation:")
    print("-" * 120)
    print(
        f"{'Stop':<10} {'Period':<15} {'Service':<10} "
        f"{'Stop Demand':<13} {'Service Freq':<14} "
        f"{'Total Freq':<12} {'Allocated Demand':<18}"
    )

    for row in result["allocated_demand"]:
        print(
            f"{row['stop']:<10} {row['period']:<15} {row['service']:<10} "
            f"{row['stop_demand']:<13.0f} "
            f"{row['service_frequency']:<14.2f} "
            f"{row['total_stop_frequency']:<12.2f} "
            f"{row['allocated_demand']:<18.2f}"
        )

    print("\nService-level demand, capacity, and overcrowding:")
    print("-" * 100)
    print(
        f"{'Service':<10} {'Period':<15} {'Frequency':<12} "
        f"{'Demand':<12} {'Capacity':<12} {'Overcrowding':<14} "
        f"{'Buses Req.':<12}"
    )

    for row in result["service_summary"]:
        print(
            f"{row['service']:<10} {row['period']:<15} "
            f"{row['frequency']:<12.2f} "
            f"{row['service_demand']:<12.2f} "
            f"{row['service_capacity']:<12.2f} "
            f"{row['overcrowding']:<14.2f} "
            f"{row['buses_required']:<12.0f}"
        )


# ============================================================
# 4. Main
# ============================================================

if __name__ == "__main__":

    instance = create_small_instance()

    result = solve_gurobi_frequency_allocation(
        *instance,
        alpha=0.40,
        beta=0.50,
        gamma=0.10,
        time_limit=120,
        verbose=True,
    )

    print_results(result)

Set parameter TimeLimit to value 120
Set parameter NonConvex to value 2
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120
NonConvex  2

Optimize a model with 93 rows, 141 columns and 333 nonzeros (Min)
Model fingerprint: 0x4686c467
Model has 60 linear objective coefficients
Model has 12 quadratic objective terms
Model has 24 quadratic constraints
Variable types: 93 continuous, 48 integer (48 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [4e+02, 3e+03]
  Objective range  [8e-04, 1e-02]
  QObjective range [5e-08, 5e-08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+03]

Presolve removed 74 rows and 110 columns
Presolve time: 0.04s
Presolved: 51 rows, 32 columns, 177 n

In [81]:
"""
Small-sized Gurobi benchmark for bi-objective bus route redesign.

Problem:
    Multiple TSP / bus route redesign.

Objectives:
    1. Minimise total route distance.
    2. Minimise demand loss.

Bi-objective method:
    Epsilon-constraint method:
        Minimise total distance subject to demand loss <= epsilon.

Node 0 = bus interchange 
Nodes 1 to n = bus stops.
"""

import math
import random
from gurobipy import Model, GRB, quicksum


# ============================================================
# 1. Generate small synthetic data
# ============================================================

def generate_data(num_stops=10, num_services=3, seed=42):
    random.seed(seed)

    depot = 0
    stops = list(range(1, num_stops + 1))
    nodes = [depot] + stops
    services = list(range(num_services))

    # Generate coordinates in a 10 km by 10 km area
    coords = {depot: (5.0, 5.0)}

    for i in stops:
        coords[i] = (
            random.uniform(0, 10),
            random.uniform(0, 10)
        )

    # Euclidean distance matrix
    distance = {}

    for i in nodes:
        for j in nodes:
            if i != j:
                xi, yi = coords[i]
                xj, yj = coords[j]
                distance[i, j] = math.sqrt((xi - xj) ** 2 + (yi - yj) ** 2)

    # Peak-period demand at each stop
    demand = {
        i: random.randint(80, 500)
        for i in stops
    }

    return nodes, stops, depot, services, coords, distance, demand


# ============================================================
# 2. Solve one epsilon-constraint model
# ============================================================

def solve_bus_route_gurobi(
    nodes,
    stops,
    depot,
    services,
    distance,
    demand,
    max_stops_per_service=4,
    max_distance_per_service=None,
    demand_loss_limit=None,
    time_limit=600,
    verbose=False
):
    model = Model("bus_route_redesign_mtsp")

    if not verbose:
        model.Params.OutputFlag = 0

    model.Params.TimeLimit = time_limit

    Z = services
    N = nodes
    V = stops
    K = max_stops_per_service

    # ------------------------------------------------------------
    # Decision variables
    # ------------------------------------------------------------

    x = {}

    for z in Z:
        for i in N:
            for j in N:
                if i != j:
                    x[z, i, j] = model.addVar(
                        vtype=GRB.BINARY,
                        name=f"x_{z}_{i}_{j}"
                    )

    y = {}

    for z in Z:
        for i in V:
            y[z, i] = model.addVar(
                vtype=GRB.BINARY,
                name=f"y_{z}_{i}"
            )

    u = {}

    for i in V:
        u[i] = model.addVar(
            vtype=GRB.BINARY,
            name=f"u_{i}"
        )

    order = {}

    for z in Z:
        for i in V:
            order[z, i] = model.addVar(
                lb=0,
                ub=K,
                vtype=GRB.CONTINUOUS,
                name=f"order_{z}_{i}"
            )

    model.update()

    # ------------------------------------------------------------
    # Routing constraints
    # ------------------------------------------------------------

    # Each service leaves depot exactly once
    for z in Z:
        model.addConstr(
            quicksum(x[z, depot, j] for j in V) == 1,
            name=f"depart_depot_{z}"
        )

    # Each service returns to depot exactly once
    for z in Z:
        model.addConstr(
            quicksum(x[z, i, depot] for i in V) == 1,
            name=f"return_depot_{z}"
        )

    # Link assignment to route inflow and outflow
    for z in Z:
        for i in V:
            model.addConstr(
                quicksum(x[z, i, j] for j in N if j != i) == y[z, i],
                name=f"outflow_link_{z}_{i}"
            )

            model.addConstr(
                quicksum(x[z, j, i] for j in N if j != i) == y[z, i],
                name=f"inflow_link_{z}_{i}"
            )

    # Each stop can be assigned to at most one service
    for i in V:
        model.addConstr(
            quicksum(y[z, i] for z in Z) <= 1,
            name=f"at_most_one_service_{i}"
        )

    # Define coverage variable
    for i in V:
        model.addConstr(
            u[i] == quicksum(y[z, i] for z in Z),
            name=f"coverage_{i}"
        )

    # Maximum number of stops per service
    for z in Z:
        model.addConstr(
            quicksum(y[z, i] for i in V) <= K,
            name=f"max_stops_service_{z}"
        )

    # Optional maximum route distance per service
    if max_distance_per_service is not None:
        for z in Z:
            model.addConstr(
                quicksum(
                    distance[i, j] * x[z, i, j]
                    for i in N
                    for j in N
                    if i != j
                ) <= max_distance_per_service,
                name=f"max_route_distance_{z}"
            )

    # ------------------------------------------------------------
    # MTZ subtour elimination
    # ------------------------------------------------------------

    for z in Z:
        for i in V:
            model.addConstr(
                order[z, i] >= y[z, i],
                name=f"order_lower_{z}_{i}"
            )

            model.addConstr(
                order[z, i] <= K * y[z, i],
                name=f"order_upper_{z}_{i}"
            )

        for i in V:
            for j in V:
                if i != j:
                    model.addConstr(
                        order[z, i] - order[z, j] + K * x[z, i, j]
                        <= K - 1,
                        name=f"mtz_{z}_{i}_{j}"
                    )

    # ------------------------------------------------------------
    # Demand loss
    # ------------------------------------------------------------

    total_demand = sum(demand[i] for i in V)

    served_demand = quicksum(
        demand[i] * u[i]
        for i in V
    )

    demand_loss = total_demand - served_demand

    if demand_loss_limit is not None:
        model.addConstr(
            demand_loss <= demand_loss_limit,
            name="demand_loss_limit"
        )

    # ------------------------------------------------------------
    # Objective: minimise total distance
    # ------------------------------------------------------------

    total_distance = quicksum(
        distance[i, j] * x[z, i, j]
        for z in Z
        for i in N
        for j in N
        if i != j
    )

    model.setObjective(total_distance, GRB.MINIMIZE)

    # ------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------

    model.optimize()

    # Important fix:
    # Do not read .X if no solution exists.
    if model.SolCount == 0:
        print("\nNo feasible incumbent solution found.")
        print("Gurobi status:", model.Status)

        if model.Status == GRB.INFEASIBLE:
            print("Model is infeasible. Computing IIS...")
            model.computeIIS()
            model.write("infeasible_model.ilp")
            print("IIS written to infeasible_model.ilp")

        return None

    # ------------------------------------------------------------
    # Extract routes safely
    # ------------------------------------------------------------

    routes = {}

    for z in Z:
        route = [depot]
        current = depot
        visited_in_route = set()

        while True:
            next_node = None

            for j in N:
                if current != j:
                    var = x.get((z, current, j))

                    if var is not None and var.X > 0.5:
                        next_node = j
                        break

            if next_node is None:
                break

            route.append(next_node)

            if next_node == depot:
                break

            if next_node in visited_in_route:
                print(f"Warning: cycle detected in route for service {z}.")
                break

            visited_in_route.add(next_node)
            current = next_node

        routes[z] = route

    selected_stops = [
        i for i in V
        if u[i].X > 0.5
    ]

    unserved_stops = [
        i for i in V
        if u[i].X < 0.5
    ]

    result = {
        "status": model.Status,
        "objective_distance": total_distance.getValue(),
        "demand_loss": demand_loss.getValue(),
        "served_demand": served_demand.getValue(),
        "total_demand": total_demand,
        "runtime": model.Runtime,
        "mip_gap": model.MIPGap if model.SolCount > 0 else None,
        "routes": routes,
        "selected_stops": selected_stops,
        "unserved_stops": unserved_stops,
        "demand": demand,
        "coords": None
    }

    return result


# ============================================================
# 3. Epsilon-constraint experiment
# ============================================================

def run_epsilon_constraint_experiment():
    nodes, stops, depot, services, coords, distance, demand = generate_data(
        num_stops=10,
        num_services=3,
        seed=7
    )

    total_demand = sum(demand[i] for i in stops)

    print("\nGenerated stop demand:")
    for i in stops:
        print(f"Stop {i}: demand = {demand[i]}")

    print("\nTotal demand:", total_demand)

    # Because 3 services x 4 stops each = maximum 12 stops,
    # all 10 stops can theoretically be served.
    epsilon_values = [
        int(total_demand * 0.60),
        int(total_demand * 0.45),
        int(total_demand * 0.30),
        int(total_demand * 0.15),
        0
    ]

    pareto_results = []

    for eps in epsilon_values:
        print(f"\nSolving with demand loss limit <= {eps}...")

        result = solve_bus_route_gurobi(
            nodes=nodes,
            stops=stops,
            depot=depot,
            services=services,
            distance=distance,
            demand=demand,
            max_stops_per_service=4,
            max_distance_per_service=None,
            demand_loss_limit=eps,
            time_limit=600,
            verbose=False
        )

        if result is not None:
            pareto_results.append(result)

    # Remove duplicate solutions
    unique_results = []
    seen = set()

    for result in pareto_results:
        key = (
            round(result["objective_distance"], 4),
            round(result["demand_loss"], 4)
        )

        if key not in seen:
            seen.add(key)
            unique_results.append(result)

    print("\n" + "=" * 90)
    print("GUROBI EPSILON-CONSTRAINT BENCHMARK RESULTS")
    print("=" * 90)

    print(
        f"{'Solution':<10} "
        f"{'Distance':<15} "
        f"{'Demand Loss':<15} "
        f"{'Served Demand':<15} "
        f"{'Runtime':<10} "
        f"{'MIP Gap':<10}"
    )

    for idx, result in enumerate(unique_results, start=1):
        print(
            f"{idx:<10} "
            f"{result['objective_distance']:<15.3f} "
            f"{result['demand_loss']:<15.1f} "
            f"{result['served_demand']:<15.1f} "
            f"{result['runtime']:<10.3f} "
            f"{result['mip_gap']:<10.5f}"
        )

    for idx, result in enumerate(unique_results, start=1):
        print("\n" + "-" * 90)
        print(f"Solution {idx}")
        print("-" * 90)

        print(f"Total distance: {result['objective_distance']:.3f}")
        print(f"Demand loss:    {result['demand_loss']:.1f}")
        print(f"Served demand:  {result['served_demand']:.1f}")
        print(f"Unserved stops: {result['unserved_stops']}")

        for z, route in result["routes"].items():
            print(f"Service {z}: {route}")

    return unique_results


# ============================================================
# 4. Main
# ============================================================

results = run_epsilon_constraint_experiment()


Generated stop demand:
Stop 1: demand = 193
Stop 2: demand = 103
Stop 3: demand = 365
Stop 4: demand = 148
Stop 5: demand = 228
Stop 6: demand = 294
Stop 7: demand = 153
Stop 8: demand = 356
Stop 9: demand = 140
Stop 10: demand = 372

Total demand: 2352

Solving with demand loss limit <= 1411...

Solving with demand loss limit <= 1058...

Solving with demand loss limit <= 705...

Solving with demand loss limit <= 352...

Solving with demand loss limit <= 0...

No feasible incumbent solution found.
Gurobi status: 3
Model is infeasible. Computing IIS...
IIS written to infeasible_model.ilp

GUROBI EPSILON-CONSTRAINT BENCHMARK RESULTS
Solution   Distance        Demand Loss     Served Demand   Runtime    MIP Gap   
1          14.699          1259.0          1093.0          0.137      0.00000   
2          16.976          1031.0          1321.0          0.365      0.00000   
3          22.526          619.0           1733.0          0.794      0.00000   
4          29.472          243.0    